In [1]:
!pip install -U transformers datasets trl peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.5 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1
  Attempting uninstall

In [2]:
import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

In [4]:
import pandas as pd
from datasets import Dataset

# 1. Load the specific Kaggle dataset
df = pd.read_csv("Video Games Data.csv")

# 2. Map the exact columns from your dataset
game_col = 'title'
release_col = 'release_date'
rating_col = 'critic_score'

# Clean the data by dropping rows missing these crucial pieces of information
df = df.dropna(subset=[game_col, release_col, rating_col])

# 3. Generate Question-Answering pairs programmatically
qa_pairs = []
for index, row in df.iterrows():
    game = row[game_col]
    date = row[release_col]
    rating = row[rating_col]

    # QA Pair 1: Release Date
    qa_pairs.append({
        "question": f"When did the game {game} come out?",
        "answer": f"{game} was released on {date}."
    })

    # QA Pair 2: Ratings/Reviews
    qa_pairs.append({
        "question": f"What are the reviews and ratings for {game}?",
        "answer": f"{game} currently holds a critic score of {rating}."
    })

# 4. Convert to Hugging Face Dataset and split for validation
qa_df = pd.DataFrame(qa_pairs)

# Shuffle and take a sample to speed up your initial fine-tuning tests.
qa_df = qa_df.sample(frac=1, random_state=42).reset_index(drop=True)
if len(qa_df) > 10000:
    qa_df = qa_df.head(10000)

dataset = Dataset.from_pandas(qa_df)
split_dataset = dataset.train_test_split(test_size=0.1)
train_data = split_dataset["train"]
val_data = split_dataset["test"]

# 5. Format prompts for the Qwen Base model
def format_prompts(examples):
    formatted_texts = []
    for q, a in zip(examples['question'], examples['answer']):
        # Qwen3.5 base model relies on standard text completion
        text = f"Question: {q}\nAnswer: {a}<|endoftext|>"
        formatted_texts.append(text)
    return {"text": formatted_texts}

train_data = train_data.map(format_prompts, batched=True)
val_data = val_data.map(format_prompts, batched=True)

print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Training samples: 9000
Validation samples: 1000


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "Qwen/Qwen3.5-2B-Base"

# Load standard Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Configure 4-bit Quantization (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# Load as a Causal Language Model for text-only QA
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,     # <-- THE FIX: Forces all unquantized weights to FP16
    device_map="auto",
    trust_remote_code=True
)
model.config.use_cache = False

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

...

model.safetensors.index.json:   0%|          | 0.00/64.5k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 4.55GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

In [6]:
# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# Configure LoRA parameters
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

model = get_peft_model(model, peft_config)

In [7]:
from trl import SFTConfig, SFTTrainer

# 1. Enforce cache disabling (Crucial for gradient checkpointing)
model.config.use_cache = False

training_args = SFTConfig(
    output_dir="./qwen-videogames-finetuned",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,
    optim="paged_adamw_32bit",
    logging_steps=5,
    learning_rate=2e-4,
    fp16=False,
    max_grad_norm=0.3,
    num_train_epochs=2,
    eval_strategy="no",
    save_strategy="epoch",
    warmup_steps=10,
    lr_scheduler_type="cosine",
    report_to="none",
    dataset_text_field="text",
    max_length=256,
    dataloader_num_workers=0,
    dataset_num_proc=1,
    gradient_checkpointing=True, # We'll keep it on, but configure it safely
    gradient_checkpointing_kwargs={"use_reentrant": False}, # <-- THE FIX: Modern PyTorch handling
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    processing_class=tokenizer,
    args=training_args,
)

# Execute Fine-Tuning
trainer.train()

# Save final artifacts
trainer.save_model("qwen-videogames-final")
tokenizer.save_pretrained("qwen-videogames-final")

print("Video Game QA Fine-tuning complete!")

Adding EOS to train dataset (num_proc=1):   0%|          | 0/9000 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=1):   0%|          | 0/9000 [00:00<?, ? examples/s]

Building labels for train dataset (num_proc=1):   0%|          | 0/9000 [00:00<?, ? examples/s]

Truncating train dataset (num_proc=1):   0%|          | 0/9000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset (num_proc=1):   0%|          | 0/9000 [00:00<?, ? examples/s…

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 248044}.


Step,Training Loss
5,2.087023
10,1.185267
15,0.802585
20,0.797244
25,0.752379
30,0.708726
35,0.760366
40,0.637913
45,0.732142
50,0.696084


Video Game QA Fine-tuning complete!
